# Genomic intervals, interactions, constraint and RNA resources

Tiny synthetic local files exercise real readers and transforms. These examples
show schema and coordinate contracts; they are not empirical biological edges
or benchmark performance. Live downloads and native-tool operations are recipes
at the end. Whole-file existence caches in these older source modules do not
provide the manifest completeness guarantees of the newer OT/GWAS APIs.


In [1]:
import gzip
import tempfile
from pathlib import Path

import polars as pl

work = Path(tempfile.mkdtemp(prefix="biodb-tutorial-"))

In [2]:
from biodb import ensembl_compara, intervals, tads

left = pl.DataFrame(
    {
        "chrom": ["chr1", "chr1"],
        "start": [100, 200],
        "end": [200, 250],
        "name": ["regionA", "regionB"],
    }
)
right = pl.DataFrame({"chrom": ["chr1"], "start": [200], "end": [220], "gene": ["GENEA"]})
joined = intervals.overlap_join(left, right)
assert joined["name"].to_list() == ["regionB"]  # abutting A is not overlap
print(joined)
print(intervals.nearest(left, right, max_distance=100))
intervals.to_bed(left, work / "regions.bed.gz", extra_columns=("name",))
assert intervals.read_bed(work / "regions.bed.gz", extra_columns=("name",)).equals(left)
domains = pl.DataFrame({"chrom": ["chr1"], "start": [100], "end": [300]})
pairs = pl.DataFrame({"chrom": ["chr1", "chr1"], "left": [110, 110], "right": [250, 300]})
print(tads.same_domain(pairs, domains))
print(tads.boundaries(domains, flank=5))
families = ensembl_compara.paralog_families(
    pl.DataFrame({"gene_a": ["ENSG1", "ENSG2"], "gene_b": ["ENSG2", "ENSG3"]})
)
assert families["family_size"].to_list() == [3, 3, 3]
print(families)

shape: (1, 7)
┌───────┬───────┬─────┬─────────┬─────────────┬───────────┬────────────┐
│ chrom ┆ start ┆ end ┆ name    ┆ start_right ┆ end_right ┆ gene_right │
│ ---   ┆ ---   ┆ --- ┆ ---     ┆ ---         ┆ ---       ┆ ---        │
│ str   ┆ i64   ┆ i64 ┆ str     ┆ i64         ┆ i64       ┆ str        │
╞═══════╪═══════╪═════╪═════════╪═════════════╪═══════════╪════════════╡
│ chr1  ┆ 200   ┆ 250 ┆ regionB ┆ 200         ┆ 220       ┆ GENEA      │
└───────┴───────┴─────┴─────────┴─────────────┴───────────┴────────────┘
shape: (2, 8)
┌───────┬───────┬─────┬─────────┬─────────────┬───────────┬────────────┬──────────┐
│ chrom ┆ start ┆ end ┆ name    ┆ start_right ┆ end_right ┆ gene_right ┆ distance │
│ ---   ┆ ---   ┆ --- ┆ ---     ┆ ---         ┆ ---       ┆ ---        ┆ ---      │
│ str   ┆ i64   ┆ i64 ┆ str     ┆ i64         ┆ i64       ┆ str        ┆ i64      │
╞═══════╪═══════╪═════╪═════════╪═════════════╪═══════════╪════════════╪══════════╡
│ chr1  ┆ 100   ┆ 200 ┆ regionA ┆ 200    

In [3]:
import numpy as np

from biodb import coessentiality

ce_cache = work / "coessentiality"
ce_cache.mkdir()
(ce_cache / "genes.txt").write_text("A\nB\nC\n")
p = np.full((3, 3), 0.5)
p[0, 1] = p[1, 0] = 1e-9
np.save(ce_cache / "GLS_p.npy", np.asfortranarray(p))
np.save(ce_cache / "GLS_sign.npy", np.ones((3, 3)))
net = coessentiality.coessential_pairs(fdr=0.01, cache_dir=ce_cache, progress=False)
assert net.select("gene_a", "gene_b").rows() == [("A", "B")]
print(net)
print(
    coessentiality.pair_p_values([("A", "B"), ("A", "UNKNOWN")], cache_dir=ce_cache, progress=False)
)

shape: (1, 4)
┌────────┬────────┬───────────┬──────┐
│ gene_a ┆ gene_b ┆ p         ┆ sign │
│ ---    ┆ ---    ┆ ---       ┆ ---  │
│ str    ┆ str    ┆ f64       ┆ f64  │
╞════════╪════════╪═══════════╪══════╡
│ A      ┆ B      ┆ 1.0000e-9 ┆ 1.0  │
└────────┴────────┴───────────┴──────┘
shape: (2, 4)
┌────────┬─────────┬───────────┬──────┐
│ gene_a ┆ gene_b  ┆ p         ┆ sign │
│ ---    ┆ ---     ┆ ---       ┆ ---  │
│ str    ┆ str     ┆ f64       ┆ f64  │
╞════════╪═════════╪═══════════╪══════╡
│ A      ┆ B       ┆ 1.0000e-9 ┆ 1.0  │
│ A      ┆ UNKNOWN ┆ null      ┆ null │
└────────┴─────────┴───────────┴──────┘


In [4]:
from biodb import corum, string

ppi_cache = work / "string"
ppi_cache.mkdir()
version, organism = string.DEFAULT_VERSION, string.DEFAULT_ORGANISM
(ppi_cache / f"{organism}.protein.physical.links.v{version}.txt.gz").write_bytes(
    gzip.compress(b"protein1 protein2 combined_score\n9606.P1 9606.P2 900\n9606.P2 9606.P1 900\n")
)
(ppi_cache / f"{organism}.protein.info.v{version}.txt.gz").write_bytes(
    gzip.compress(
        b"#string_protein_id\tpreferred_name\tprotein_size\tannotation\n9606.P1\tGENEA\t10\tSynthetic A\n9606.P2\tGENEB\t20\tSynthetic B\n"
    )
)
print(string.physical_ppi_edges(cache_dir=ppi_cache))
# CORUM co-complex membership is a different relation from STRING physical PPI.
corum_cache = work / "corum" / "5.3"
corum_cache.mkdir(parents=True)
(corum_cache / "corum_humanComplexes.txt").write_text(
    "complex_id\tcomplex_name\torganism\tsubunits_gene_name\n1\tSynthetic trio\tHuman\tGENEA;GENEB;GENEC\n"
)
complexes = corum.complex_pairs(version="5.3", cache_dir=work / "corum", progress=False)
assert complexes.height == 3
print(complexes)

  gene_a gene_b  combined_score  score
0  GENEA  GENEB             900    0.9
shape: (3, 5)
┌────────┬────────┬────────────┬────────────────┬──────────┐
│ gene_a ┆ gene_b ┆ complex_id ┆ complex_name   ┆ subunits │
│ ---    ┆ ---    ┆ ---        ┆ ---            ┆ ---      │
│ str    ┆ str    ┆ i64        ┆ str            ┆ i64      │
╞════════╪════════╪════════════╪════════════════╪══════════╡
│ GENEA  ┆ GENEB  ┆ 1          ┆ Synthetic trio ┆ 3        │
│ GENEA  ┆ GENEC  ┆ 1          ┆ Synthetic trio ┆ 3        │
│ GENEB  ┆ GENEC  ┆ 1          ┆ Synthetic trio ┆ 3        │
└────────┴────────┴────────────┴────────────────┴──────────┘


In [5]:
from biodb import constraint_tracks, rfam

# GPN-Star published positions are 1-based; derived start is zero-based.
revision = constraint_tracks.GPN_STAR_REVISION
path = (
    work
    / "gpn-star"
    / revision
    / "data"
    / constraint_tracks.GPN_STAR_MODELS["m447"]
    / "entropy"
    / "entropy_chr1.parquet"
)
path.parent.mkdir(parents=True)
pl.DataFrame(
    {"chrom": ["1", "1"], "pos": [101, 102], "ref": ["A", "C"], "entropy_calibrated": [0.4, 1.0]}
).write_parquet(path)
constraint = constraint_tracks.load_gpn_star("1", positions=[101], cache_dir=work, progress=False)
assert constraint["start"][0] == 100 and constraint["chrom"][0] == "chr1"
print(constraint)
# Parse a real gzip Stockholm-format seed fixture without invoking Infernal.
seed_dir = work / "rfam" / "15.1"
seed_dir.mkdir(parents=True)
(seed_dir / "Rfam.seed.gz").write_bytes(
    gzip.compress(b"# STOCKHOLM 1.0\n#=GF AC RF00001\nseqA ACG\nseqB ACG\n#=GC SS_cons (.)\n//\n")
)
sequences, structure = rfam.seed_structure(
    "RF00001", release="15.1", cache_dir=work / "rfam", progress=False
)
assert structure == "(.)"
print("Rfam seed:", sequences, structure)

shape: (1, 5)
┌───────┬─────┬─────┬────────────────────┬───────┐
│ chrom ┆ pos ┆ ref ┆ entropy_calibrated ┆ start │
│ ---   ┆ --- ┆ --- ┆ ---                ┆ ---   │
│ str   ┆ i64 ┆ str ┆ f64                ┆ i64   │
╞═══════╪═════╪═════╪════════════════════╪═══════╡
│ chr1  ┆ 101 ┆ A   ┆ 0.4                ┆ 100   │
└───────┴─────┴─────┴────────────────────┴───────┘
Rfam seed: {'seqA': 'ACG', 'seqB': 'ACG'} (.)


## Live production recipes — not executed

Use an assembly-consistent coordinate system before combining sources. BED,
TAD, CRISPRi and bigWig coordinates are zero-based half-open; published GPN-Star
`pos` is one-based. Each edge family has a different interpretation and must
retain source provenance.

```python
from biodb import traitgym, encode_re2g, orphanet
files = tads.list_domain_files(biosample="K562", assembly="GRCh38")
domains = tads.load_domains(files["accession"][0])
paralogs = ensembl_compara.paralog_pairs(release="116", min_identity=50)
complexes = corum.complex_pairs(version="5.3", organism="Human")
ppi = string.physical_ppi_edges(min_combined_score=700)
coessential = coessentiality.coessential_pairs(fdr=0.01, positive_only=True)
# A subset filter does not redefine the published genomewide FDR universe.

positives_and_negatives = traitgym.load_variants("mendelian_traits_matched_9")
feature_names = traitgym.list_features("mendelian_traits_matched_9")
features = traitgym.load_feature("mendelian_traits_matched_9", feature_names[0])
crispri = encode_re2g.crispri_pairs(cell_type="K562", valid_only=True)
prevalence = orphanet.load_prevalence()
k_by_orpha = orphanet.prevalence_map(prevalence, validated_only=True)

# Requires biodb[bigwig]; reads selected intervals over HTTPS, not a whole download.
bw = constraint_tracks.open_track("phylop241")
try:
    scores = constraint_tracks.score_intervals(bw, left, stat="mean")
finally:
    bw.close()
# GPN-Star chromosome Parquet downloads can be gigabytes despite positions=.
# scores = constraint_tracks.load_gpn_star("chr1", positions=[101])

# Requires external Infernal cmscan/cmpress; no executable/model runs in this notebook.
# result = rfam.scan_fasta("genome.fa", genome_size_mb=6200, release="15.1")
# hits = rfam.load_hits(result.tblout)
```

TraitGym and ENCODE-rE2G pin Hub revisions by default. Preserve benchmark
labels, negatives and cell context; they are not interchangeable evidence.
Orphanet's `k` is a per-individual prevalence-band estimate, not a gene score or
an incidence rate. Check the source's geography and validation status.
